In [1]:
import numpy as np
from ngsolve import *
from ngsolve.webgui import Draw
from netgen.occ import *
from ngsolve.solvers import NewtonMinimization

In [26]:
r = 0.25
l = 3

bar = MoveTo(-r,0).Rectangle(2*r, l).Face()
bar.edges.Min(Y).name="rotation"
bar.edges.Min(Y).maxh=r/10
bar.faces.maxh=0.5/2
bar = bar.Rotate(Axis((0.0, 0, 0), (0, 0, 1)), 180)
bar = bar.Rotate(Axis((0.0, 0, 0), (0, 0, 1)), 45)
bar.name = "pendulum"
geo = bar
geo = OCCGeometry(geo, dim=2)
mesh = Mesh(geo.GenerateMesh(maxh=0.1))
Draw(mesh)

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

BaseWebGuiScene

In [21]:
E = 2.1e11
nu = 0.2
rho = 7800
thickness = 0.3
g=9.81

lam = (E*nu)/((1+nu)*(1-2*nu))
mu = E/(2*(1+nu))

def C(u):
    F = Id(u.dim) + Grad(u)
    return F.trans * F
    
def neo_hookean( C, u):
    return 0.5*mu*(Trace(C-Id(u.dim)) + 2*mu/lam*Det(C)**(-lam/2/mu)-1)


In [22]:
V = VectorH1(mesh, order=2)
Q = NumberSpace(mesh, definedon=mesh.Boundaries("rotation"))

fes = V * Q**2

(u, q), (v, p) = fes.TnT()

In [23]:
gf_u = GridFunction(fes)
gf_v = GridFunction(fes)
gf_a = GridFunction(fes)

gf_uold = GridFunction(fes)
gf_vold = GridFunction(fes)
gf_aold = GridFunction(fes)

In [25]:
bfa = BilinearForm(fes)

bfa += Variation(neo_hookean(C(u), u)*dx).Compile()

bfa += (InnerProduct(u, p) + InnerProduct(v, q)) * ds('rotation')

tau = Parameter(0.01)
vel_new = 2/tau * (u-gf_uold.components[0]) - gf_vold.components[0]
acc_new = 2/tau * (vel_new-gf_vold.components[0]) - gf_aold.components[0]

rhoA = rho * thickness
bfa += rhoA * InnerProduct(acc_new, v) * dx
bfa += InnerProduct(CF((0, rhoA * g)), v) * dx("pendulum")